# Gold: fact_order_payments

## Import Helper Functions

In [1]:
from src.config_loader import load_config
from src.spark_sql_magic import sql
from src.gold.helper import get_changed_customer_ids, get_changed_order_ids
from src.gold.facts.order_payments import build_fact_order_payments, build_fact_order_payments_incremental, validate_fact_order_payments
from src.watermark import get_last_commit_ts, get_effective_watermark
from src.writers import overwrite_table, replace_by_key

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
SILVER_NAMESPACE = cfg["general"]["namespaces"]["silver"]
GOLD_NAMESPACE = cfg["general"]["namespaces"]["gold"]

cfg_order_payments= cfg["gold"]["fact_order_items"]
SOURCE_TABLE = cfg_order_payments["source_table"]
TARGET_TABLE = cfg_order_payments["target_table"]
ORDERS_TABLE = cfg_order_payments["orders_table"]
CUSTOMERS_TABLE = cfg_order_payments["customers_table"]
DATE_TABLE = cfg_order_payments["date_table"]
BUFFER_HOURS = cfg_order_payments["buffer_hours"]
KEY_COLUMNS = cfg_order_payments["key_columns"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["gold_order_payments"])
        .getOrCreate()
)

## Run Pipeline

In [4]:
def run_fact_order_payments_pipeline(spark):
    print("[START] fact_order_items pipeline")
    last_commit_ts = get_last_commit_ts(spark, TARGET_TABLE)
    print(f"[INFO] last_commit_ts = {last_commit_ts}")

    if last_commit_ts is None:
        print("[INFO] first run → full rebuild")
        df = build_fact_order_payments(
            spark,
            SOURCE_TABLE,
            ORDERS_TABLE, 
            CUSTOMERS_TABLE,
            DATE_TABLE
        )
        validate_fact_order_payments(df)
        overwrite_table(df, TARGET_TABLE)

    effective_ts = get_effective_watermark(last_commit_ts, BUFFER_HOURS)
    changed_order_ids = get_changed_order_ids(spark, effective_ts)
    changed_customer_ids = get_changed_customer_ids(spark, effective_ts)

    if changed_customer_ids.isEmpty() and changed_order_ids.isEmpty():
        print("[INFO] no changes detected → skip")
        return

    print("[INFO] changes detected → incremental run")
    df = build_fact_order_payments_incremental(
        spark,
        SOURCE_TABLE,
        ORDERS_TABLE, 
        CUSTOMERS_TABLE, 
        DATE_TABLE,
        changed_order_ids,
        changed_customer_ids,
    )
    validate_fact_order_payments(df)
    replace_by_key(spark, df, TARGET_TABLE, KEY_COLUMNS)
    print("[END] incremental update complete")

In [5]:
# if __name__ == "__main__":
#     from pyspark.sql import SparkSession

#     spark = SparkSession.builder.getOrCreate()
run_fact_order_payments_pipeline(spark)

[START] fact_order_items pipeline
[INFO] last_commit_ts = 2026-04-03 04:34:06.766000


[INFO] changes detected → incremental run


{"ts": "2026-04-03 04:35:30.190", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `op`.`payment_sequential` cannot be resolved. Did you mean one of the following? [`op`.`created_at`, `o`.`created_at`, `dp`.`is_weekend`, `op`.`order_item_id`, `o`.`order_status`]. SQLSTATE: 42703", "context": {"file": "line 33 in cell [5]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o198.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `op`.`payment_sequential` cannot be resolved. Did you mean one of the following? [`op`.`created_at`, `o`.`created_at`, `dp`.`is_weekend`, `op`.`order_item_id`, `o`.`order_status`]. SQLSTATE: 42703;\n'Project [order_id#95, 'op.payment_sequential, 'op.payment_type

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `op`.`payment_sequential` cannot be resolved. Did you mean one of the following? [`op`.`created_at`, `o`.`created_at`, `dp`.`is_weekend`, `op`.`order_item_id`, `o`.`order_status`]. SQLSTATE: 42703;
'Project [order_id#95, 'op.payment_sequential, 'op.payment_type, 'op.payment_installments, 'op.payment_value, customer_sk#199, customer_id#216, date_sk#206 AS order_purchase_date_sk#428, order_purchase_timestamp#218]
+- Join LeftOuter, (to_date(order_purchase_timestamp#218, None, Some(UTC), true) = ds#205)
   :- Join LeftOuter, (((customer_id#216 = customer_id#187) AND (order_purchase_timestamp#218 >= effective_from#196)) AND ((order_purchase_timestamp#218 < effective_to#197) OR isnull(effective_to#197)))
   :  :- Project [order_id#95, order_item_id#96, product_id#97, seller_id#98, shipping_limit_date#99, price#100, freight_value#101, created_at#102, updated_at#103, _op#104, cdc_ts_ms#1

## Sanity Check

In [ ]:
%%sql
SHOW TABLES IN polaris.gold;

In [ ]:
%%sql
SELECT * FROM polaris.gold.fact_order_payments
LIMIT 10

In [ ]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop() 